In [1]:
# Bootstrap : rendre src/ importable et ancrer les chemins sur la racine projet
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from src.paths import MAIN_DATASET, MODELS_DIR
from src.preprocessing import prepare_features_and_target, split_data, scale_features

df = pd.read_csv(MAIN_DATASET)
random_state = 42  # Toujours fixer le seed pour la reproductibilité
print(f"Dataset chargé : {df.shape}")

# Sépare features / cible via le module (norm_squared exclue par défaut —
# voir notebooks 07/08 pour la discussion complète du leakage)
X, y = prepare_features_and_target(df, include_norm_squared=False)
print(f"Features : {X.shape}, cible : {y.shape}")


Dataset chargé : (10000, 11)
Features : (10000, 8), cible : (10000,)


In [2]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)

print(f"Train: {len(X_train)}")
print(f"Val: {len(X_val)}")
print(f"Test: {len(X_test)}")

# Vérification de la stratification
print(f"\nClasse 1 dans train: {(y_train==1).sum()/len(y_train):.1%}")
print(f"Classe 1 dans val: {(y_val==1).sum()/len(y_val):.1%}")
print(f"Classe 1 dans test: {(y_test==1).sum()/len(y_test):.1%}")

Train: 6000
Val: 2000
Test: 2000

Classe 1 dans train: 50.0%
Classe 1 dans val: 50.0%
Classe 1 dans test: 50.0%


In [3]:
# Baseline : régression logistique sur features standardisées.
# NOTE : ce notebook valide le PIPELINE (split -> scaling -> fit), pas la
# science — l'évaluation honnête du problème est dans le notebook 08.
X_train_s, X_val_s, X_test_s, scaler = scale_features(
    X_train, X_val, X_test, method="standard", save_scaler=False
)

model = LogisticRegression(max_iter=2000, random_state=random_state)
model.fit(X_train_s, y_train)

val_acc = accuracy_score(y_val, model.predict(X_val_s))
print(f"Accuracy validation : {val_acc:.4f}")
print("(faible attendu : coordonnées brutes seules, frontière non linéaire — cf. nb 08)")


Accuracy validation : 0.5670
(faible attendu : coordonnées brutes seules, frontière non linéaire — cf. nb 08)


In [4]:
import joblib

y_test_pred = model.predict(X_test_s)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Accuracy sur TEST : {test_accuracy:.4f}")

# Sauvegarde du modèle baseline, ancrée sur la racine projet (src/paths.py)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODELS_DIR / "baseline_model.joblib"
joblib.dump(model, model_path)
print(f"Modèle sauvegardé : {model_path.name} ({model_path.stat().st_size} octets)")
print("Pipeline de preprocessing validé de bout en bout.")


Accuracy sur TEST : 0.5435
Modèle sauvegardé : baseline_model.joblib (1263 octets)
Pipeline de preprocessing validé de bout en bout.
